**Eye-Tracking & Neurocognitive Intelligence | Eye-Tracking Foundations & Signal Analysis | Presented by: Reza Saadatyar (2026)**

**04 | Gaze Features and Fixation Scanpaths**

![Conceptual workflow for stage 04](../../images/04_gaze_features_and_scanpaths.svg)

**Objective:** Aggregate identified events into per-record features and inspect fixation-centroid paths.

**Dataset:** Notebook 02 prepared samples and Notebook 03 event intervals.

**Notebook scope:** Produces feature tables and scanpaths; does not infer reading comprehension or AOI mappings without stimulus geometry.

**Execution status:** Source code prepared. No original-data results are claimed until this notebook is executed with the specified release.


**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eye_tracking_foundations")

# ---------------------------- 2. Standard Library ----------------------------
from pathlib import Path

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

# ------------------------ 5. Study Research Utilities ------------------------
from core.stage_outputs import ensure_stage_inputs
from gaze_source import read_stage
from gaze_features import record_features, feature_dictionary

# --------------------- 6. Output and Presentation Setup ----------------------
output_dir = root / "eye_tracking_foundations" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
plot_style(
    font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5,
    label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(10, 4),
)
print("Current outputs:", output_dir)


**02. Recover Prepared Samples and Detected Events**

Bring back only the two required upstream tables. Existing stage outputs
are reused as-is; one explicitly named prior-stage ZIP per stage can be
restored in a fresh Colab runtime.


In [ ]:
# ============= 02. Recover Prepared Samples and Detected Events ==============

ensure_stage_inputs(output_dir, 2, ("02_preprocessed_samples.csv",))
ensure_stage_inputs(output_dir, 3, ("03_events.csv",))
prepared = read_stage(output_dir / "02_preprocessed_samples.csv",
                      ("record_id","participant_id","valid_for_motion","x_smooth","y_smooth"))
events = read_stage(output_dir / "03_events.csv",
                    ("record_id","participant_id","method","event_type","duration_ms"))


**03. Calculate Record-Level Features**

Features describe the selected method (I-VT), not physiological ground
truth. Scanpath length is the sum of distances between successive fixation
centroids in dva. Only valid, positive, available pupil measurements enter
the optional uncalibrated pupil summary.


In [ ]:
# ==================== 03. Calculate Record-Level Features ====================

features = pd.DataFrame([
    record_features(part, events[events.record_id.eq(rid)])
    for rid, part in prepared.groupby("record_id",sort=True)
])
features.to_csv(output_dir / "04_features_by_record.csv", index=False)
feature_dictionary().to_csv(output_dir / "04_feature_dictionary.csv",index=False)
display(features)


**04. Visualise an Illustrative Fixation-Centroid Path**

This is a sequence of detected fixation centroids, not an annotated word
scanpath or behavioural judgement. Plot one source record with available
fixations and show acquisition order.


In [ ]:
# =========== 04. Visualise an Illustrative Fixation-Centroid Path ============

rid = str(prepared.record_id.iloc[0])
fix = events[(events.record_id==rid) & (events.method=="IVT") &
             (events.event_type=="fixation")].head(100)
fig, ax = plt.subplots(figsize=(7,5))
if len(fix):
    ax.plot(fix.mean_x_dva, fix.mean_y_dva, lw=0.9, marker="o", markersize=3)
    ax.scatter(fix.mean_x_dva.iloc[0], fix.mean_y_dva.iloc[0],marker="s",s=50,
               label="First selected fixation")
else:
    ax.text(0.5,0.5,"No qualifying fixations in selected record",
            ha="center",transform=ax.transAxes)
ax.set(xlabel="Gaze x (dva)",ylabel="Gaze y (dva)", title=f"I-VT fixation sequence: {rid}")
ax.invert_yaxis()
ax.legend(loc="best") if len(fix) else None
fig.tight_layout()
save_figure(fig,output_dir / "04_scanpath.png")
plt.show()


**05. Review Feature Coverage**

The number of independent people is different from the number of records,
fixations or samples. Notebook 05 will aggregate across repeated records
before uncertainty calculations.


In [ ]:
# ======================== 05. Review Feature Coverage ========================

print("Feature records:",len(features),"Unique participants:",features.participant_id.nunique())
print("Unavailable values by feature:")
display(features.isna().sum().rename("missing"))


**06. Export Named Current Features**

Save stage 04 for later statistical and integrated reporting notebooks.


In [ ]:
# ===================== 06. Export Named Current Features =====================

download_outputs_from_colab("eye_tracking_foundations/output", "04")
